In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
from sqlalchemy import create_engine

from src.config import postgres_config

In [2]:
connection_url = (
    f"postgresql+psycopg2://{postgres_config.user}:"
    f"{postgres_config.password}@{postgres_config.host}:"
    f"{postgres_config.port}/{postgres_config.database}"
)

engine = create_engine(connection_url)

In [3]:
df = pd.read_sql("SELECT * FROM training_data", engine)
df.head()

,id,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type,cover_type,group_number,batch_number,inserted_at
0,1,3008,336,11,210,22,671,194,225,169,1024,0.0,1.0,0,8,2,2026-03-14 18:42:00.811826
1,2,3028,71,26,42,8,1020,238,181,59,2312,0.0,0.0,1,8,2,2026-03-14 18:42:00.811826
2,3,3167,165,13,124,19,3528,231,244,141,1624,0.0,0.0,1,8,2,2026-03-14 18:42:00.811826
3,4,3142,48,14,0,0,1875,224,209,116,2023,0.0,0.0,1,8,2,2026-03-14 18:42:00.811826
4,5,3157,343,11,85,8,870,197,223,164,1983,1.0,0.0,0,8,2,2026-03-14 18:42:00.811826


In [4]:
df.shape

(5810, 17)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5810 entries, 0 to 5809
Data columns (total 17 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   id                                  5810 non-null   int64         
 1   elevation                           5810 non-null   int64         
 2   aspect                              5810 non-null   int64         
 3   slope                               5810 non-null   int64         
 4   horizontal_distance_to_hydrology    5810 non-null   int64         
 5   vertical_distance_to_hydrology      5810 non-null   int64         
 6   horizontal_distance_to_roadways     5810 non-null   int64         
 7   hillshade_9am                       5810 non-null   int64         
 8   hillshade_noon                      5810 non-null   int64         
 9   hillshade_3pm                       5810 non-null   int64         
 10  horizontal_distance_to_fire_points 

In [6]:
df["cover_type"].value_counts()

cover_type
1    4070
0    1740
Name: count, dtype: int64

In [7]:
X = df.drop(
    columns=[
        "id",
        "cover_type",
        "group_number",
        "batch_number",
        "inserted_at"
    ]
)

y = df["cover_type"]

In [8]:
X.head()

,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type
0,3008,336,11,210,22,671,194,225,169,1024,0.0,1.0
1,3028,71,26,42,8,1020,238,181,59,2312,0.0,0.0
2,3167,165,13,124,19,3528,231,244,141,1624,0.0,0.0
3,3142,48,14,0,0,1875,224,209,116,2023,0.0,0.0
4,3157,343,11,85,8,870,197,223,164,1983,1.0,0.0


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [10]:
numeric_features = [
    "elevation",
    "aspect",
    "slope",
    "horizontal_distance_to_hydrology",
    "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am",
    "hillshade_noon",
    "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]

categorical_features = [
    "wilderness_area",
    "soil_type",
]

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, classification_report

In [15]:
models = {
    "logistic_regression": LogisticRegression(max_iter=2000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "gradient_boosting": GradientBoostingClassifier()
}

In [18]:
trained_models = {}

for name, clf in models.items():
    
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ]
    )
    
    pipe.fit(X_train, y_train)
    
    preds = pipe.predict(X_test)
    
    print("\n", name)
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))
    
    trained_models[name] = pipe


 logistic_regression
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       338
           1       1.00      1.00      1.00       824

    accuracy                           1.00      1162
   macro avg       1.00      1.00      1.00      1162
weighted avg       1.00      1.00      1.00      1162


 random_forest
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       338
           1       1.00      1.00      1.00       824

    accuracy                           1.00      1162
   macro avg       1.00      1.00      1.00      1162
weighted avg       1.00      1.00      1.00      1162


 gradient_boosting
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       338
           1       1.00      1.00      1.00       824

    accuracy                           1.00      1162
   macro avg       1.00     

In [19]:
import joblib
from pathlib import Path

In [20]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

In [21]:
for name, model in trained_models.items():
    
    model_path = models_dir / f"{name}.joblib"
    
    joblib.dump(model, model_path)
    
    print(f"Saved {name} → {model_path}")

Saved logistic_regression → ../models/logistic_regression.joblib
Saved random_forest → ../models/random_forest.joblib
Saved gradient_boosting → ../models/gradient_boosting.joblib


In [1]:
from pathlib import Path
from minio import Minio

In [2]:
minio_client = Minio(
    endpoint="localhost:9000",
    access_key="minio_admin",
    secret_key="minio_admin_password",
    secure=False
)

In [3]:
bucket_name = "models"

In [4]:
found = minio_client.bucket_exists(bucket_name)
print("Bucket exists:", found)

Bucket exists: True


In [5]:
models_dir = Path("../models")

for model_path in models_dir.glob("*.joblib"):
    object_name = model_path.name

    minio_client.fput_object(
        bucket_name=bucket_name,
        object_name=object_name,
        file_path=str(model_path)
    )

    print(f"Uploaded {object_name} to MinIO bucket '{bucket_name}'")

Uploaded logistic_regression.joblib to MinIO bucket 'models'
Uploaded gradient_boosting.joblib to MinIO bucket 'models'
Uploaded random_forest.joblib to MinIO bucket 'models'
